# Set up

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import random
import time
import os
import gc
import copy
from pathlib import Path
from tqdm.auto import tqdm
from contextlib import nullcontext
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.amp import autocast, GradScaler

import sys
sys.path.append('/scratch/bng/cartbind/code/MIND_models/QuantNets/autoencoders')
from models import AgeGuidedAutoencoder, AgeGuidedLoss
from metrics import calc_r2_corr

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Configuration

In [ ]:
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'
region_dir = base_dir / 'region_names'

# Load the column renamer to be applied down the line
rename_df = pd.read_csv(region_dir / 'col_renames_dnanexus.csv')
rename_dict = dict(zip(rename_df['datafield_code'], rename_df['datafield_name']))

# Sub-directories for the new outputs
run_name = 'AE_ElasticNet_new_apr24'
weights_dir = base_dir / f'models_AE_elasticnet_dnanexus/{run_name}_weights_scaling_law'
results_dir = base_dir / f'models_AE_elasticnet_dnanexus/{run_name}_scaling_law_results'
predictions_dir = base_dir / f'models_AE_elasticnet_dnanexus/{run_name}_predictions_scaling_law'
ae_curves_dir = base_dir / f'models_AE_elasticnet_dnanexus/{run_name}_ae_training_curves'

os.makedirs(weights_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(ae_curves_dir, exist_ok=True)

targets = {
    'GF': ('GF', 'p20016_i2'),
    'PAL': ('PAL', 'p20197_i2'),
    'DSST': ('DSST', 'p23324_i2'),
    'TMT': ('TMT', 'p6350_i2'),
}

data_configs = {
    'FC25': (region_dir / 'FC25_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'FC100': (region_dir / 'FC100_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'MIND': (region_dir / 'MIND_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),   
}
# sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']
sample_sizes = ['all']

# --- OPTUNA HYPERPARAMS ---
ae_hyperparams = {
    'FC25': {
        '1_hide': {'hidden_dims': [128], 'latent_dim': 64, 'lr': 0.00508689, 'recon_weight': 0.93, 'ae_weight_decay': 5.463816, 'age_weight_decay': 0.0450122, 'age_predictor_hidden_dims': [32, 16, 8, 4], 'age_predictor_dropout': 0.25, 'epochs': 150},   # trial 286
        # '2_hide': {'hidden_dims': [128, 64], 'latent_dim': 32, 'lr': 1e-3, 'recon_weight': 0.5, 'ae_weight_decay': 1e-4, 'age_weight_decay': 1e-4, 'age_predictor_hidden_dims': [16, 8], 'age_predictor_dropout': 0.1, 'epochs': 150}
    },
    'FC100': {
        '1_hide': {'hidden_dims': [1024], 'latent_dim': 512, 'lr': 0.000534843, 'recon_weight': 0.88, 'ae_weight_decay': 1.481096, 'age_weight_decay': 0.000229702, 'age_predictor_hidden_dims': [4], 'age_predictor_dropout': 0.7, 'epochs': 150},   # trial 296
        # '2_hide': {'hidden_dims': [512], 'latent_dim': 256, 'lr': 0.001581248146663570, 'recon_weight': 0.8556561897363870, 'ae_weight_decay': 9.659428526695E-05, 'age_weight_decay': 1.56089378355989E-06, 'age_predictor_hidden_dims': [16], 'age_predictor_dropout': 0.5910979365703270, 'epochs': 150}
    },
    'MIND': {
        '1_hide': {'hidden_dims': [1024], 'latent_dim': 512, 'lr': 0.0001293066, 'recon_weight': 0.99, 'ae_weight_decay': 1.56703, 'age_weight_decay': 0.0000839535, 'age_predictor_hidden_dims': [256, 32], 'age_predictor_dropout': 0.55, 'epochs': 150},   # trial 286
        # '2_hide': {'hidden_dims': [1024, 512], 'latent_dim': 256, 'lr': 1e-3, 'recon_weight': 0.5, 'ae_weight_decay': 1e-4, 'age_weight_decay': 1e-4, 'age_predictor_hidden_dims': [128, 64], 'age_predictor_dropout': 0.1, 'epochs': 150}
    }
}

# Training functions

In [ ]:
def extract_features(model, loader, device):
    model.eval()
    latents, recons, age_preds = [], [], []
    with torch.no_grad():
        for batch_x, batch_age in loader:
            batch_x = batch_x.to(device)
            x_hat, z, age_pred = model(batch_x)
            latents.append(z.cpu().float().numpy())
            recons.append(x_hat.cpu().float().numpy())
            age_preds.append(age_pred.cpu().float().numpy())
            
    return np.concatenate(latents, axis=0), np.concatenate(recons, axis=0), np.concatenate(age_preds, axis=0)

In [ ]:
def train_ae_and_extract_latent(X_train_br, age_train, X_test_br, age_test, ae_params, early_stop_patience=15):
    # Ensure reproducible initialization per fold
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    X_scaler, age_scaler = StandardScaler(), StandardScaler()
    x_train_scaled_full = X_scaler.fit_transform(X_train_br.values)
    x_test_scaled = X_scaler.transform(X_test_br.values)
    
    age_train_scaled_full = age_scaler.fit_transform(age_train.values.reshape(-1, 1)).flatten()
    age_test_scaled = age_scaler.transform(age_test.values.reshape(-1, 1)).flatten()

    x_ae_train, x_ae_val, age_ae_train, age_ae_val = train_test_split(
        x_train_scaled_full, age_train_scaled_full, test_size=0.15, random_state=seed
    )

    batch_size = 256
    
    train_loader = DataLoader(TensorDataset(torch.tensor(x_ae_train, dtype=torch.float32), torch.tensor(age_ae_train, dtype=torch.float32)), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(x_ae_val, dtype=torch.float32), torch.tensor(age_ae_val, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    
    full_train_extract_loader = DataLoader(TensorDataset(torch.tensor(x_train_scaled_full, dtype=torch.float32), torch.tensor(age_train_scaled_full, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    test_extract_loader = DataLoader(TensorDataset(torch.tensor(x_test_scaled, dtype=torch.float32), torch.tensor(age_test_scaled, dtype=torch.float32)), batch_size=batch_size, shuffle=False)

    try:
        model = AgeGuidedAutoencoder(
            input_dim=X_train_br.shape[1], latent_dim=ae_params['latent_dim'], hidden_dims=ae_params['hidden_dims'],
            age_predictor_hidden_dims=ae_params['age_predictor_hidden_dims'], age_predictor_dropout=ae_params['age_predictor_dropout']
        ).to(device)
        
        criterion = AgeGuidedLoss(recon_weight=ae_params['recon_weight'], age_weight=1.0 - ae_params['recon_weight'])
        
        ae_decay, age_decay, no_decay = [], [], []
        for name, param in model.named_parameters():
            if 'bn' in name or 'bias' in name: no_decay.append(param)
            elif 'age_predictor' in name: age_decay.append(param)
            else: ae_decay.append(param)

        optimizer = torch.optim.AdamW([
            {'params': ae_decay, 'weight_decay': ae_params.get('ae_weight_decay', 1e-4)},
            {'params': age_decay, 'weight_decay': ae_params.get('age_weight_decay', 1e-4)},
            {'params': no_decay, 'weight_decay': 0.0}
        ], lr=ae_params['lr'])

        # --- LR SCHEDULER ADDITION ---
        num_epochs = ae_params['epochs']
        warmup_epochs = max(1, int(num_epochs * 0.05))
        warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
        decay_epochs = num_epochs - warmup_epochs
        cosine_scheduler = CosineAnnealingLR(optimizer, T_max=decay_epochs)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
        
        scaler = GradScaler(device.type) if device.type == 'cuda' else None
        
        curve_records = []
        best_val_loss = float('inf')
        early_stop_counter = 0
        best_model_state = None
        
        for epoch in range(num_epochs):
            model.train()
            epoch_loss = 0
            for batch_x, batch_age in train_loader:
                batch_x, batch_age = batch_x.to(device), batch_age.to(device)
                optimizer.zero_grad()
                
                context_manager = autocast(device.type) if scaler else nullcontext()
                with context_manager:
                    x_hat, _, age_pred = model(batch_x)
                    loss = criterion(batch_x, x_hat, batch_age, age_pred)
                    
                if scaler:
                    scaler.scale(loss['total_loss']).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss['total_loss'].backward()
                    optimizer.step()
                    
                epoch_loss += loss['total_loss'].item() * batch_x.size(0)
                
            train_loss = epoch_loss / len(train_loader.dataset)
            
            # --- STEP THE SCHEDULER EVERY EPOCH ---
            scheduler.step()
            
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_x, batch_age in val_loader:
                    batch_x, batch_age = batch_x.to(device), batch_age.to(device)
                    
                    context_manager = autocast(device.type) if scaler else nullcontext()
                    with context_manager:
                        x_hat, _, age_pred = model(batch_x)
                        loss = criterion(batch_x, x_hat, batch_age, age_pred)
                    val_loss += loss['total_loss'].item() * batch_x.size(0)
                    
            val_loss = val_loss / len(val_loader.dataset)
            curve_records.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                early_stop_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                early_stop_counter += 1
                
            if early_stop_counter >= early_stop_patience: break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        z_train, _, _ = extract_features(model, full_train_extract_loader, device)
        z_test, x_hat_test, age_pred_test = extract_features(model, test_extract_loader, device)

        x_test_inv = X_scaler.inverse_transform(x_hat_test)
        age_pred_inv = age_scaler.inverse_transform(age_pred_test.reshape(-1, 1)).flatten()
        
        ae_recon_r2 = r2_score(X_test_br.values, x_test_inv)
        ae_recon_r2_corr = sum(calc_r2_corr(X_test_br.values[:, i], x_test_inv[:, i]) for i in range(x_test_inv.shape[1])) / x_test_inv.shape[1]
        ae_age_r2 = r2_score(age_test.values, age_pred_inv)
        ae_age_r2_corr = calc_r2_corr(age_test.values, age_pred_inv)

        return z_train, z_test, ae_recon_r2, ae_recon_r2_corr, ae_age_r2, ae_age_r2_corr, curve_records

    # --- AGGRESSIVE GARBAGE COLLECTION BLOCK ---
    finally:
        # Ignore NameErrors if any failed to initialize before crashing
        try:
            del model, optimizer, scheduler, scaler, train_loader, val_loader, full_train_extract_loader, test_extract_loader
        except NameError:
            pass
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def analysis(X, y, df, data_name, target_name, arc_name, sample_size, brain_regions, demographic_vars, ae_params, n_splits=10):
    os.makedirs(weights_dir / target_name, exist_ok=True)
    os.makedirs(predictions_dir / target_name, exist_ok=True)
    
    preds_path = predictions_dir / target_name / f'{run_name}_{arc_name}_preds_{data_name}_{target_name}_{sample_size}.csv'
    weights_path = weights_dir / target_name / f'{run_name}_{arc_name}_weights_{data_name}_{target_name}_{sample_size}.csv'
    
    outer_cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    outer_mae, outer_rmse, outer_r2, outer_r2_corr = [], [], [], []
    fold_alphas, fold_l1_ratios, fold_nonzeros = [], [], []
    ae_recons, ae_recons_corr, ae_ages, ae_ages_corr = [], [], [], []
    weight_records = []
    
    continuous_dvars = [c for c in demographic_vars if c not in ['sex', 'assessment_centre']]
    categorical_dvars = [c for c in demographic_vars if c in ['sex', 'assessment_centre']]

    print(f"\nEvaluating Base={data_name}, Target={target_name}, Arc={arc_name}, N={sample_size}")
    start_time = time.time()
    
    for fold, (train_idx, test_idx) in enumerate(tqdm(outer_cv.split(X, y), total=n_splits, desc=f"CV Folds", leave=False), start=1):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        age_col = 'age' if 'age' in df.columns else 'p21003_i2'
        age_train, age_test = df.iloc[train_idx][age_col], df.iloc[test_idx][age_col]
        
        z_train, z_test, recon_r2, recon_r2_corr, age_r2, age_r2_corr, curve_df = train_ae_and_extract_latent(
            X_train[brain_regions], age_train, 
            X_test[brain_regions], age_test, 
            ae_params
        )
        
        ae_recons.append(recon_r2); ae_recons_corr.append(recon_r2_corr)
        ae_ages.append(age_r2); ae_ages_corr.append(age_r2_corr)
        
        pd.DataFrame(curve_df).to_csv(ae_curves_dir / f"{data_name}_{arc_name}_{target_name}_n{sample_size}_f{fold}_curve.csv", index=False)

        z_cols = [f"z_{i}" for i in range(z_train.shape[1])]
        X_train_elnet = pd.concat([X_train[demographic_vars].copy().reset_index(drop=True), pd.DataFrame(z_train, columns=z_cols)], axis=1)
        X_test_elnet = pd.concat([X_test[demographic_vars].copy().reset_index(drop=True), pd.DataFrame(z_test, columns=z_cols)], axis=1)

        preprocessor = ColumnTransformer(transformers=[
            ('num', StandardScaler(), continuous_dvars + z_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_dvars),
        ])
        
        pipe = TransformedTargetRegressor(
            regressor=make_pipeline(
                preprocessor,
                ElasticNetCV(
                    l1_ratio=[0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99,1.0],
                    alphas=np.logspace(-5,2,15), cv=10, max_iter=40000, random_state=seed, n_jobs=-1
                )
            ),
            transformer=StandardScaler()
        )
        
        pipe.fit(X_train_elnet, y_train.values.reshape(-1, 1))
        y_pred = pipe.predict(X_test_elnet).ravel()
        
        fold_df = pd.DataFrame({'fold': fold, 'eid': y_test.index, 'actual': y_test.values, 'predicted': y_pred})
        fold_df.to_csv(preds_path, mode='a', header=(fold==1), index=False)
        
        elnet_model = pipe.regressor_.named_steps['elasticnetcv']
        
        # Track ElasticNet configuration metrics
        fold_alphas.append(elnet_model.alpha_)
        fold_l1_ratios.append(elnet_model.l1_ratio_)
        fold_nonzeros.append(np.sum(elnet_model.coef_ != 0))

        # Extract the fitted preprocessor from the pipeline clone
        fitted_preprocessor = pipe.regressor_.named_steps['columntransformer']
        transformed_columns = fitted_preprocessor.get_feature_names_out()
        
        clean_col_names = [col.replace('num__', '').replace('cat__', '') for col in transformed_columns]
        
        weight_df = pd.DataFrame({'feature': clean_col_names, 'weight': elnet_model.coef_, 'fold': fold})
        weight_records.append(weight_df)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        r2_corr = calc_r2_corr(y_test, y_pred)

        outer_mae.append(mae)
        outer_rmse.append(rmse)
        outer_r2.append(r2)
        outer_r2_corr.append(r2_corr)

        print(f'  Fold {fold:02d} • MAE={mae:.3f} • RMSE={rmse:.3f} • R²={r2:.3f} • R²(corr)={r2_corr:.3f} '
              f'• α={elnet_model.alpha_:.4g} • l1_ratio={elnet_model.l1_ratio_:.2f} '
              f'• AE Recon R²(corr)={recon_r2_corr:.3f} • AE Age R²(corr)={age_r2_corr:.3f}')

        # Hard memory cleaning of the fold ML models
        del pipe, preprocessor, elnet_model, z_train, z_test, X_train_elnet, X_test_elnet
        gc.collect()
        
    print(f'  Summary Config • ElasticNet Mean R²(corr)={np.mean(outer_r2_corr):.3f} '
          f'• Mean AE Recon R²(corr)={np.mean(ae_recons_corr):.3f} • Mean AE Age R²(corr)={np.mean(ae_ages_corr):.3f}')

    pd.concat(weight_records).to_csv(weights_path, index=False)
    
    elapsed_time = time.time() - start_time

    return {
        'arc_name':                  arc_name,
        'mean_mae':                  np.mean(outer_mae),
        'mean_rmse':                 np.mean(outer_rmse),
        'mean_r2':                   np.mean(outer_r2),
        'std_r2':                    np.std(outer_r2),
        'mean_r2_corr':              np.mean(outer_r2_corr),
        'std_r2_corr':               np.std(outer_r2_corr),
        'ae_test_recon_r2':          np.mean(ae_recons),
        'std_ae_test_recon_r2':      np.std(ae_recons),
        'ae_test_recon_r2_corr':     np.mean(ae_recons_corr),
        'std_ae_test_recon_r2_corr': np.std(ae_recons_corr),
        'ae_test_age_r2':            np.mean(ae_ages),
        'std_ae_test_age_r2':        np.std(ae_ages),
        'ae_test_age_r2_corr':       np.mean(ae_ages_corr),
        'std_ae_test_age_r2_corr':   np.std(ae_ages_corr),
        'mean_alpha':                np.mean(fold_alphas),
        'mean_l1_ratio':             np.mean(fold_l1_ratios),
        'mean_nonzero_coefs':        np.mean(fold_nonzeros),
        'elapsed_time_sec':          elapsed_time
    }


# Scaling law loop

In [ ]:
# Main scaling entry
for target_name, (test_key, score_col) in targets.items():
    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    df_full = pd.read_csv(data_file, index_col=0)
    
    # Pre-map DataFrame renaming before iterating through data configs
    df_full = df_full.rename(columns=rename_dict)
    
    # Rename 'score_col' mapping correctly if it's one of the standardized variables
    if score_col in rename_dict:
        score_col = rename_dict[score_col]

    for data_name, (regions_file, demographic_vars) in data_configs.items():
        with open(regions_file, 'r') as f:
            brain_regions = [line.strip() for line in f]
            
        # Rename the configurations variables themselves from the list matching
        brain_regions = [rename_dict.get(br, br) for br in brain_regions]
        demographic_vars = [rename_dict.get(dv, dv) for dv in demographic_vars]
            
        all_vars = demographic_vars + brain_regions

        for arc_name in ['1_hide', '2_hide']:
            ae_params = ae_hyperparams[data_name][arc_name]

            for sample_size in sample_sizes:
                eid_file = splits_dir / target_name / (f'{target_name}_all_eids.txt' if sample_size == 'all' else f'{target_name}_eids_{sample_size}.txt')
                if not eid_file.exists(): continue
                    
                sample_eids = np.loadtxt(eid_file, dtype=int)
                df = df_full[df_full['eid'].isin(sample_eids)] if 'eid' in df_full.columns else df_full[df_full.index.isin(sample_eids)]

                # Prep Matrices
                X, y = df[all_vars], df[score_col]

                metrics = analysis(X, y, df, data_name, target_name, arc_name, sample_size, brain_regions, demographic_vars, ae_params)

                row_df = pd.DataFrame([{
                    'target_name': target_name, 'data_name': data_name, 
                    'arc_name': arc_name, 'sample_size': sample_size, 'actual_n': len(df), **metrics
                }])
                
                results_file = results_dir / f'scaling_law_results_AE_ElasticNet_{target_name}.csv'
                row_df.to_csv(str(results_file), mode='a', header=not results_file.exists(), index=False)